# Pilot 2021 — Extraction, segmentation, mapping

**Scope:** November 2021 P1 + P2 exams (memos support verification)  
**Reference year:** 2022 (`pilot-2022-v1`)  
**Taxonomy:** frozen v1  

## Pipeline change from 2022
- `diagram_present`, `load_bearing_diagram`, `extraction_method`, `data_quality` from the start
- OCR issues register before accept
- Vision supplement only for **load-bearing** rows after first text pass
- 10-row sample tick required

## Gates
- Source PDFs present and non-empty
- Classification complete
- Segmentation: ~10 main questions per paper
- Mapping coverage ≥ 85% (then inheritance + load-bearing pass)
- Sample verification ≥ 9/10 before tag

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re

import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

RAW_EXAMS = PROJECT_ROOT / "data" / "raw" / "exams"
RAW_MEMOS = PROJECT_ROOT / "data" / "raw" / "memos"
TEXT_DIR = PROJECT_ROOT / "data" / "interim" / "extracted_text" / "2021"
SEG_DIR = PROJECT_ROOT / "data" / "interim" / "segmented" / "2021"
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2021"
PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / "2021"

for d in [TEXT_DIR, SEG_DIR, MAP_DIR, PILOT_DOC]:
    d.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "p1_exam": RAW_EXAMS / "2021_nov_p1_exam_maths.pdf",
    "p2_exam": RAW_EXAMS / "2021_nov_p2_exam_maths.pdf",
    "p1_memo": RAW_MEMOS / "2021_nov_p1_memo_maths.pdf",
    "p2_memo": RAW_MEMOS / "2021_nov_p2_memo_maths.pdf",
}

print("PROJECT_ROOT:", PROJECT_ROOT)
for k, p in SOURCES.items():
    ok = p.exists() and p.stat().st_size > 0
    size = p.stat().st_size if p.exists() else 0
    print(f"  {k}: {'OK' if ok else 'MISSING'}  {size:>10}  {p.name}")

PROJECT_ROOT: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence
  p1_exam: OK      322738  2021_nov_p1_exam_maths.pdf
  p2_exam: OK      464562  2021_nov_p2_exam_maths.pdf
  p1_memo: OK      647532  2021_nov_p1_memo_maths.pdf
  p2_memo: OK      935380  2021_nov_p2_memo_maths.pdf


In [2]:
import pdfplumber

rows = []
for name, path in SOURCES.items():
    if not path.exists():
        rows.append({"doc": name, "exists": False})
        continue
    with pdfplumber.open(path) as pdf:
        n = len(pdf.pages)
        sample = ""
        for page in pdf.pages[: min(3, n)]:
            sample += page.extract_text() or ""
        cpp = len(sample) / max(1, min(3, n))
        rows.append({
            "doc": name,
            "exists": True,
            "pages": n,
            "chars_per_page": round(cpp, 1),
            "has_text_layer": cpp > 200,
            "ocr_required": cpp <= 200,
        })

cls = pd.DataFrame(rows)
cls.to_csv(TEXT_DIR / "ocr_classification.csv", index=False)
display(cls)
print("Saved:", TEXT_DIR / "ocr_classification.csv")

,doc,exists,pages,chars_per_page,has_text_layer,ocr_required
0,p1_exam,True,10,0.0,False,True
1,p2_exam,True,14,0.0,False,True
2,p1_memo,True,16,760.3,True,False
3,p2_memo,True,24,886.0,True,False


Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\interim\extracted_text\2021\ocr_classification.csv


In [3]:
import pdfplumber

def extract_pdfplumber(path: Path) -> str:
    parts = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, 1):
            t = page.extract_text() or ""
            parts.append(f"\n===== Page {i} =====\n{t}")
    return "\n".join(parts)

for name in ["p1_memo", "p2_memo"]:
    path = SOURCES[name]
    text = extract_pdfplumber(path)
    out = TEXT_DIR / f"{name}.txt"
    out.write_text(text, encoding="utf-8")
    ok = ("MATHEMATICS" in text.upper()) or ("WISKUNDE" in text.upper()) or ("QUESTION" in text.upper())
    print(f"{name}: chars={len(text)} marker_ok={ok} -> {out.name}")

p1_memo: chars=12661 marker_ok=True -> p1_memo.txt
p2_memo: chars=20525 marker_ok=True -> p2_memo.txt


In [4]:
import os
from pdf2image import convert_from_path
import pytesseract

POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"
TESSDATA = r"C:\Users\Administrator\tessdata"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
os.environ["TESSDATA_PREFIX"] = TESSDATA
OCR_CONFIG = r"--tessdata-dir C:\Users\Administrator\tessdata --psm 6"

for name in ["p1_exam", "p2_exam"]:
    pdf_path = SOURCES[name]
    print(f"OCR: {name} ...")
    pages = convert_from_path(str(pdf_path), dpi=300, poppler_path=POPPLER_PATH)
    parts = []
    for i, img in enumerate(pages, 1):
        t = pytesseract.image_to_string(img, lang="eng", config=OCR_CONFIG)
        parts.append(f"\n===== Page {i} =====\n{t}")
        print(f"  page {i}/{len(pages)} chars={len(t)}")
    full = "\n".join(parts)
    out = TEXT_DIR / f"{name}.txt"
    out.write_text(full, encoding="utf-8")
    ok = any(k in full.upper() for k in ["MATHEMATICS", "QUESTION", "NOVEMBER"])
    print(f"  saved {out.name} total_chars={len(full)} marker_ok={ok}")
print("OCR complete.")

OCR: p1_exam ...
  page 1/10 chars=295
  page 2/10 chars=886
  page 3/10 chars=710
  page 4/10 chars=1225
  page 5/10 chars=381
  page 6/10 chars=796
  page 7/10 chars=1467
  page 8/10 chars=523
  page 9/10 chars=1360
  page 10/10 chars=765
  saved p1_exam.txt total_chars=8618 marker_ok=True
OCR: p2_exam ...
  page 1/14 chars=289
  page 2/14 chars=835
  page 3/14 chars=1202
  page 4/14 chars=1048
  page 5/14 chars=1006
  page 6/14 chars=725
  page 7/14 chars=712
  page 8/14 chars=587
  page 9/14 chars=491
  page 10/14 chars=363
  page 11/14 chars=529
  page 12/14 chars=273
  page 13/14 chars=608
  page 14/14 chars=814
  saved p2_exam.txt total_chars=9780 marker_ok=True
OCR complete.


In [5]:
for name in ["p1_exam", "p1_memo", "p2_exam", "p2_memo"]:
    p = TEXT_DIR / f"{name}.txt"
    if not p.exists():
        print("MISSING", name)
        continue
    t = p.read_text(encoding="utf-8", errors="replace")
    print(
        name,
        "chars=", len(t),
        "MATH=", "MATHEMATICS" in t.upper(),
        "NOV=", "NOVEMBER" in t.upper(),
        "Q1=", ("QUESTION 1" in t.upper() or "QUESTION1" in t.upper()),
    )

p1_exam chars= 8618 MATH= True NOV= True Q1= True
p1_memo chars= 12661 MATH= True NOV= True Q1= False
p2_exam chars= 9780 MATH= True NOV= True Q1= True
p2_memo chars= 20525 MATH= True NOV= True Q1= False


In [6]:
def clean_text(t: str) -> str:
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t


def segment_exam(text: str, document_id: str, paper: str) -> pd.DataFrame:
    text = clean_text(text)
    q_pat = re.compile(r"(?i)(?:^|\n)\s*QUESTION\s*(\d{1,2})\b")
    matches = list(q_pat.finditer(text))
    records = []
    if not matches:
        return pd.DataFrame(records)

    for i, m in enumerate(matches):
        qnum = int(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()

        sub_pat = re.compile(
            r"(?m)(?P<sub>\d+\.\d+(?:\.\d+)*)\s+(?P<body>.*?)(?=(?:\n\s*\d+\.\d+(?:\.\d+)*\s+)|\Z)",
            re.DOTALL,
        )
        subs = list(sub_pat.finditer(block))
        if not subs:
            marks = None
            mm = re.search(r"[\(\[]\s*(\d{1,2})\s*[\)\]]\s*$", block.strip())
            if mm:
                marks = int(mm.group(1))
            records.append({
                "document_id": document_id,
                "year": 2021,
                "paper": paper,
                "question_number": qnum,
                "subquestion": str(qnum),
                "marks": marks,
                "question_text": block[:2000],
                "source_type": "exam",
                "segmentation_status": "no_sub_markers",
            })
            continue

        for s in subs:
            sub = s.group("sub").strip()
            body = s.group("body").strip()
            marks = None
            mm = re.search(r"[\(\[]\s*(\d{1,2})\s*[\)\]]\s*$", body)
            if mm:
                marks = int(mm.group(1))
                body = body[: mm.start()].strip()
            records.append({
                "document_id": document_id,
                "year": 2021,
                "paper": paper,
                "question_number": qnum,
                "subquestion": sub,
                "marks": marks,
                "question_text": body[:2000],
                "source_type": "exam",
                "segmentation_status": "ok" if marks is not None else "missing_marks",
            })
    return pd.DataFrame(records)


all_rows = []
for paper, fname, did in [
    ("P1", "p1_exam.txt", "2021_nov_p1_exam_maths"),
    ("P2", "p2_exam.txt", "2021_nov_p2_exam_maths"),
]:
    text = (TEXT_DIR / fname).read_text(encoding="utf-8", errors="replace")
    df = segment_exam(text, did, paper)
    # Cap absurd single-item marks
    if len(df) and "marks" in df.columns:
        df.loc[df["marks"] > 15, "segmentation_status"] = "mark_capped"
        df.loc[df["marks"] > 15, "marks"] = pd.NA
    out = SEG_DIR / f"{paper.lower()}_questions.csv"
    df.to_csv(out, index=False)
    n_q = df["question_number"].nunique() if len(df) else 0
    marks_sum = df["marks"].sum(min_count=1)
    missing = int(df["marks"].isna().sum()) if len(df) else 0
    print(f"{paper}: rows={len(df)} main_q={n_q} marks_sum={marks_sum} missing_marks={missing}")
    all_rows.append(df)

exams = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
exams.to_csv(SEG_DIR / "exam_questions_2021.csv", index=False)
print("Combined rows:", len(exams))
print(exams.groupby("paper").size())
print(exams["segmentation_status"].value_counts())
display(exams[["paper", "question_number", "subquestion", "marks"]].head(15))

P1: rows=57 main_q=12 marks_sum=120.0 missing_marks=19
P2: rows=62 main_q=11 marks_sum=93.0 missing_marks=26
Combined rows: 119
paper
P1    57
P2    62
dtype: int64
segmentation_status
ok                73
missing_marks     41
mark_capped        3
no_sub_markers     2
Name: count, dtype: int64


,paper,question_number,subquestion,marks
0,P1,1,1.1,NaN
1,P1,1,1.1.1,3.0
2,P1,1,1.1.2,3.0
3,P1,1,1.1.3,4.0
4,P1,1,1.1.4,4.0
5,P1,1,1.2,6.0
6,P1,1,1.3,NaN
7,P1,2,2.1,2.0
8,P1,2,2.2,2.0
9,P1,2,2.3,NaN


In [7]:
print((TEXT_DIR / "p1_exam.txt").read_text(encoding="utf-8", errors="replace")[:800])
print("---")
print(exams["segmentation_status"].value_counts())


===== Page 1 =====
“=i basic education
{ Uap) y ) Department:
LG) / Basic Education
REPUBLIC OF SOUTH AFRICA
NATIONAL
SENIOR CERTIFICATE
GRADE 12
" MATHEMATICS P1 i
" NOVEMBER 2021 ;
i Ps
MARKS: 150
TIME: 3 hours
This question paper consists of 9 pages and 1 information sheet.
Copyright reserved Please turn over


===== Page 2 =====
Mathematics P1 2 DBE/November 2021
NSC

INSTRUCTIONS AND INFORMATION

Read the following instructions carefully before answering the questions.

1. This question paper consists of 12 questions.

2. Answer ALL the questions.

3. Number the answers correctly according to the numbering system used in this
question paper.

4, Clearly show ALL calculations, diagrams, graphs, etc. that you have used in
determining your answers.

5. Answers only will NOT necessarily 
---
segmentation_status
ok                73
missing_marks     41
mark_capped        3
no_sub_markers     2
Name: count, dtype: int64


In [8]:
def norm(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).lower()
    return s.replace("θ", "theta").replace("π", "pi")


def map_topic(text: str, paper: str = ""):
    t = norm(text)
    paper = str(paper).upper()

    if re.search(
        r"f'\s*\(|dy/dx|d/dx|derivative|differentiate|second derivative|"
        r"concave|inflection|stationary|integrat|gradient of the curve",
        t,
    ):
        return "Calculus", "symbolic", "D_CALC", "high"

    if re.search(
        r"\bsin\b|\bcos\b|\btan\b|\bsec\b|\bcosec\b|\bcot\b|"
        r"identit|reduction|general solution|trigonomet|theta",
        t,
    ):
        return "Trigonometry", "keyword", "D_TRIG", "high"

    if re.search(
        r"gradient of|equation of the line|midpoint|distance formula|"
        r"circle with centre|radius of|analytical|coordinates of the centre|"
        r"\bcircles?\b|\(x\s*[−\-–]",
        t,
    ):
        return "Analytical Geometry", "keyword", "D_AGEO", "high"

    if re.search(
        r"theorem|cyclic|similar triangle|proportion|"
        r"tangent to the circle|semicircle|euclidean|prove that.*angle",
        t,
    ):
        return "Euclidean Geometry", "keyword", "D_EUCL", "high"

    if re.search(
        r"boxplot|box-and-whisker|standard deviation|variance|ogive|"
        r"histogram|quartile|interquartile|percentile|scatter",
        t,
    ):
        return "Statistics", "keyword", "D_STAT", "high"

    if re.search(
        r"probability|independent|mutually exclusive|tree diagram|venn|p\s*\(|random",
        t,
    ):
        return "Probability", "keyword", "D_PROB", "high"

    if re.search(
        r"arithmetic|geometric|sequence|series|common difference|"
        r"common ratio|constant ratio|consecutive terms|sigma|"
        r"\bterm of\b|first term|number of consecutive",
        t,
    ):
        return "Number Patterns & Sequences", "keyword", "D_SEQ", "high"

    if re.search(
        r"compound interest|simple interest|present value|future value|"
        r"annuity|depreciation|effective rate|nominal|loan|invest",
        t,
    ):
        return "Finance", "keyword", "D_FIN", "high"

    if re.search(
        r"parabola|hyperbola|exponential|logarithmic|asymptote|"
        r"inverse function|axis of symmetry|turning point|sketch|"
        r"x-intercept|y-intercept|range of|domain",
        t,
    ):
        return "Functions & Graphs", "keyword", "D_FUNC", "high"

    if re.search(
        r"quadratic|factoris|factoriz|simultaneous|inequalit|surd|"
        r"exponent|logarithm|nature of the roots|discriminant|"
        r"solve for|=\s*0|show that|determine the values of x",
        t,
    ):
        return "Algebra & Equations", "keyword", "D_ALG", "high"

    if re.search(r"[a-z0-9\)\]]\s*=\s*[a-z0-9\-\+]", t) and len(t) < 80:
        if paper == "P1":
            return "Algebra & Equations", "ocr_equation", "D_ALG_EQ", "medium"

    if len(t) < 20:
        return "Unmapped", "prior_weak", "D_SHORT", "low"

    return "Unmapped", "no_match", "D_NONE", "low"


df = exams.copy()
rows = []
for _, r in df.iterrows():
    topic, method, rule_id, conf = map_topic(r.get("question_text", ""), r.get("paper", ""))
    row = r.to_dict()
    row.update({
        "topic": topic,
        "mapping_method": method,
        "rule_id": rule_id,
        "mapping_confidence": conf,
        "taxonomy_version": "v1_frozen",
        "extraction_method": "ocr_text",
        "vision_description": None,
        "vision_verification_status": None,
        "diagram_present": False,
        "load_bearing_diagram": False,
        "data_quality": "text_usable",
        "mapped_at": datetime.now(timezone.utc).isoformat(),
    })
    rows.append(row)

mapped = pd.DataFrame(rows)


def inherit_topic(group: pd.DataFrame) -> pd.DataFrame:
    known = group[group["topic"] != "Unmapped"]
    if known.empty:
        return group
    high = known[known["mapping_confidence"] == "high"]
    base = high if len(high) else known
    winner = base["topic"].value_counts().index[0]
    mask = group["topic"] == "Unmapped"
    group.loc[mask, "topic"] = winner
    group.loc[mask, "mapping_method"] = "inherited"
    group.loc[mask, "rule_id"] = "D_INHERIT_Q"
    group.loc[mask, "mapping_confidence"] = "medium"
    return group


mapped = mapped.groupby(["paper", "question_number"], group_keys=False).apply(inherit_topic)

# Circle post-fix (P2-only safety)
def retag_circles(row):
    t = norm(row.get("question_text", ""))
    if re.search(r"circle|centre|center|radius|\(x\s*[−\-–]|\(x-a\)", t):
        if row["topic"] in ("Functions & Graphs", "Unmapped"):
            row["topic"] = "Analytical Geometry"
            row["mapping_method"] = "post_fix_circle"
            row["rule_id"] = "D_AGEO_CIRCLE"
            row["mapping_confidence"] = "high"
    return row


mapped = mapped.apply(retag_circles, axis=1)

total = len(mapped)
mapped_n = int((mapped["topic"] != "Unmapped").sum())
coverage = mapped_n / total if total else 0

print(f"Rows: {total}")
print(f"Mapped: {mapped_n} ({coverage:.1%})")
print(f"Unmapped: {total - mapped_n}")
print()
display(mapped["topic"].value_counts().rename("count").to_frame())
print()
display(pd.crosstab(mapped["paper"], mapped["topic"]))
print()
display(mapped["mapping_method"].value_counts().rename("count").to_frame())

Rows: 119
Mapped: 111 (93.3%)
Unmapped: 8



,count
topic,
Analytical Geometry,29
Algebra & Equations,22
Trigonometry,14
Statistics,13
Functions & Graphs,12
Number Patterns & Sequences,8
Unmapped,8
Probability,6
Finance,5


KeyError: 'paper'

In [9]:
# Restore core columns if lost
mapped = mapped.reset_index(drop=True)
base = exams.reset_index(drop=True)
for col in ["paper", "question_number", "subquestion", "document_id", "marks", "question_text"]:
    if col not in mapped.columns and col in base.columns:
        mapped[col] = base[col]

print("Columns:", list(mapped.columns))
print()
if "paper" in mapped.columns:
    display(pd.crosstab(mapped["paper"], mapped["topic"]))
print()
print("Unmapped:")
display(
    mapped[mapped["topic"] == "Unmapped"][
        [c for c in ["paper", "question_number", "subquestion", "question_text"] if c in mapped.columns]
    ]
)

out = MAP_DIR / "question_topic_map_2021.csv"
mapped.to_csv(out, index=False)
print("Saved:", out)
print(f"Coverage: {coverage:.1%}")

Columns: ['document_id', 'year', 'subquestion', 'marks', 'question_text', 'source_type', 'segmentation_status', 'topic', 'mapping_method', 'rule_id', 'mapping_confidence', 'taxonomy_version', 'extraction_method', 'vision_description', 'vision_verification_status', 'diagram_present', 'load_bearing_diagram', 'data_quality', 'mapped_at', 'paper', 'question_number']



topic,Algebra & Equations,Analytical Geometry,Calculus,Euclidean Geometry,Finance,Functions & Graphs,Number Patterns & Sequences,Probability,Statistics,Trigonometry,Unmapped
paper,,,,,,,,,,,
P1,22,2,1,0,5,11,8,6,0,1,1
P2,0,27,0,1,0,1,0,0,13,13,7



Unmapped:


,paper,question_number,subquestion,question_text
49,P1,11,11,QUESTION 11\nAfter travelling a distance of 20...
102,P2,8,8.1,Prove that QS = 5 tanx
103,P2,8,8.2,Prove that the length of QT =10sinx
104,P2,8,8.3,Calculate the area of ATQR if TQR = 70° and x=...
105,P2,9,9.1,Give areason why ST =TR.
106,P2,9,9.2,"Calculate, giving reasons, the size of:"
107,P2,9,9.2.1,"S,"
108,P2,9,9.2.2,"S, (2)\n[5]\nCopyright reserved Please turn ov..."


Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\2021\question_topic_map_2021.csv
Coverage: 93.3%


In [10]:
# Broad diagram_present by typical NSC structure — refine after you skim papers
DIAGRAM_Q = {
    ("P1", "4"), ("P1", "5"), ("P1", "7"), ("P1", "8"), ("P1", "9"), ("P1", "10"),
    ("P2", "1"), ("P2", "2"), ("P2", "3"), ("P2", "4"), ("P2", "6"),
    ("P2", "7"), ("P2", "8"), ("P2", "9"), ("P2", "10"),
}

def qnum_str(r):
    try:
        return str(int(r["question_number"]))
    except Exception:
        return str(r.get("question_number", ""))

def flag_diagram(r):
    key = (str(r.get("paper", "")), qnum_str(r))
    return key in DIAGRAM_Q

mapped["diagram_present"] = mapped.apply(flag_diagram, axis=1)
mapped["load_bearing_diagram"] = False  # strict set after sample + OCR register
mapped["data_quality"] = mapped["diagram_present"].map(
    {True: "text_usable", False: "text_usable"}  # refined after OCR issues
)
# Default extraction already ocr_text from Cell 6

print("diagram_present:", mapped["diagram_present"].value_counts().to_dict())
mapped.to_csv(MAP_DIR / "question_topic_map_2021.csv", index=False)
print("Re-saved with diagram_present")

diagram_present: {True: 82, False: 37}
Re-saved with diagram_present


In [11]:
sample = mapped.sample(n=min(10, len(mapped)), random_state=21)[
    [c for c in ["paper", "question_number", "subquestion", "topic", "mapping_method", "question_text"] if c in mapped.columns]
]
display(sample)
sample.to_csv(PILOT_DOC / "sample_verification_candidates.csv", index=False)

# Empty OCR register template — you fill from paper skim (like 2022)
ocr_stub = pd.DataFrame(columns=["paper", "page", "subq", "issue", "severity"])
ocr_stub.to_csv(PILOT_DOC / "ocr_issues_2021.csv", index=False)

log = """# 2021 Pilot — Manual sample verification

Tick against the **PDF**, not OCR text only.

| # | Paper | Subq | Assigned topic | Verdict (OK/WRONG/UNSURE) | Correct if wrong |
|---|-------|------|----------------|---------------------------|------------------|
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |
| 4 |  |  |  |  |  |
| 5 |  |  |  |  |  |
| 6 |  |  |  |  |  |
| 7 |  |  |  |  |  |
| 8 |  |  |  |  |  |
| 9 |  |  |  |  |  |
| 10 |  |  |  |  |  |

## Summary
- Correct: ___ / 10
- Watch: Calculus and Euclidean counts look low — include at least one of each in the 10 if possible
- Pilot decision: ACCEPT with limitations / REVISE

## Load-bearing (strict)
List subquestions that cannot be answered without the diagram/table:
- 
"""
(PILOT_DOC / "sample_verification.md").write_text(log, encoding="utf-8")
print("Wrote sample stubs in docs/pilot/2021/")

,paper,question_number,subquestion,topic,mapping_method,question_text
90,P2,5,5.2,Trigonometry,keyword,Prove the identity: —————__————- = 2cosx-1 (4)...
106,P2,9,9.2,Unmapped,no_match,"Calculate, giving reasons, the size of:"
36,P1,8,8.3,Finance,inherited,Thabo wanted to save R450 000 as a deposit to ...
66,P2,2,2.2,Statistics,inherited,Determine the equation of the least squares re...
91,P2,5,5.3,Trigonometry,inherited,Given: sin36°=./1— p’\nWithout using a calcula...
23,P1,6,6.1,Algebra & Equations,inherited,Calculate the value of k.
7,P1,2,2.1,Algebra & Equations,inherited,| Calculate the value of x.
42,P1,9,2.2,Algebra & Equations,inherited,D_| -——+| — 4\n7 x 2 =| oe\n{12]\nCopyright re...
111,P2,10,10.3,Analytical Geometry,keyword,"Prove, giving reasons, that AD is a tangent to..."
64,P2,1,1.3.2,Statistics,inherited,Describe the skewness of the data. (i)\n[10}\n...


Wrote sample stubs in docs/pilot/2021/


In [12]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
MAP = PROJECT_ROOT / "data" / "processed" / "mapped" / "2021" / "question_topic_map_2021.csv"
mapped = pd.read_csv(MAP)

cols = [c for c in [
    "subquestion", "topic", "mapping_method", "mapping_confidence", "question_text"
] if c in mapped.columns]

for paper, q in [("P1", 9), ("P1", 10), ("P1", 11), ("P2", 9), ("P2", 10), ("P2", 11)]:
    m = mapped[
        (mapped["paper"].astype(str) == paper)
        & (mapped["question_number"].astype(str).str.replace(r"\.0$", "", regex=True) == str(q))
    ]
    print(f"\n=== {paper} Q{q}  rows={len(m)} ===")
    if len(m) == 0:
        print("  NO ROWS — segmentation gap")
        continue
    display(m[cols])
    # show short text samples
    for _, r in m.head(4).iterrows():
        txt = str(r.get("question_text", ""))[:120].replace("\n", " ")
        print(f"  {r.get('subquestion')}: [{r.get('topic')}|{r.get('mapping_method')}] {txt}")


=== P1 Q9  rows=4 ===


,subquestion,topic,mapping_method,mapping_confidence,question_text
39,9.1,Algebra & Equations,ocr_equation,medium,Determine f ‘(x) from first principles if it i...
40,9.2,Algebra & Equations,inherited,medium,Determine:
41,9.2.1,Algebra & Equations,ocr_equation,medium,Y i¢ y =4x° —6x* +3x (3)\ndx\nVx (1y
42,2.2,Algebra & Equations,inherited,medium,D_| -——+| — 4\n7 x 2 =| oe\n{12]\nCopyright re...


  9.1: [Algebra & Equations|ocr_equation] Determine f ‘(x) from first principles if it is given that f(x) =2x? —3x.
  9.2: [Algebra & Equations|inherited] Determine:
  9.2.1: [Algebra & Equations|ocr_equation] Y i¢ y =4x° —6x* +3x (3) dx Vx (1y
  2.2: [Algebra & Equations|inherited] D_| -——+| — 4 7 x 2 =| oe {12] Copyright reserved Please turn over  ===== Page 8 ===== Mathematics P1 8 DBE/November 202

=== P1 Q10  rows=6 ===


,subquestion,topic,mapping_method,mapping_confidence,question_text
43,10.1,Algebra & Equations,keyword,high,Show that a=-1 and b=6.
44,10.2,Algebra & Equations,inherited,medium,Calculate the coordinates of A.
45,10.3,Algebra & Equations,inherited,medium,Write down the values of x for which A is:
46,10.3.1,Algebra & Equations,inherited,medium,Increasing
47,10.3.2,Calculus,symbolic,high,Concave down
48,10.4,Analytical Geometry,keyword,high,For which values of & will -—(x—1)’+6(x-1)?-k=...


  10.1: [Algebra & Equations|keyword] Show that a=-1 and b=6.
  10.2: [Algebra & Equations|inherited] Calculate the coordinates of A.
  10.3: [Algebra & Equations|inherited] Write down the values of x for which A is:
  10.3.1: [Algebra & Equations|inherited] Increasing

=== P1 Q11  rows=1 ===


...

  9.1: [Unmapped|no_match] Give areason why ST =TR.
  9.2: [Unmapped|no_match] Calculate, giving reasons, the size of:
  9.2.1: [Unmapped|prior_weak] S,
  9.2.2: [Unmapped|no_match] S, (2) [5] Copyright reserved Please turn over  ===== Page 11 ===== Mathematics/P2 11 DBE/November 2021 NSC

=== P2 Q10  rows=4 ===


,subquestion,topic,mapping_method,mapping_confidence,question_text
109,10.1,Analytical Geometry,inherited,medium,Give areason why AF = FE.
110,10.2,Analytical Geometry,inherited,medium,"Determine, giving reasons, the size of M, in t..."
111,10.3,Analytical Geometry,keyword,high,"Prove, giving reasons, that AD is a tangent to..."
112,10.4,Analytical Geometry,inherited,medium,"Given that CF = 6 units and AB = 24 units, cal..."


  10.1: [Analytical Geometry|inherited] Give areason why AF = FE.
  10.2: [Analytical Geometry|inherited] Determine, giving reasons, the size of M, in terms of x.
  10.3: [Analytical Geometry|keyword] Prove, giving reasons, that AD is a tangent to the circle passing through A, C and F.
  10.4: [Analytical Geometry|inherited] Given that CF = 6 units and AB = 24 units, calculate, giving reasons, the length of AE. (5) [13] Copyright reserved Plea

=== P2 Q11  rows=6 ===


,subquestion,topic,mapping_method,mapping_confidence,question_text
113,11.1,Analytical Geometry,keyword,high,"In the diagram, chords DE, EF and DF are drawn..."
114,11.2,Analytical Geometry,keyword,high,"In the diagram, PK is a tangent to the circle ..."
115,11.2.1,Euclidean Geometry,keyword,high,"Prove, giving reasons, that:\n@) &,=NNIL (4)\n..."
116,11.2.2,Analytical Geometry,inherited,medium,"Prove, giving reasons, that ALKN ||| AKSM."
117,11.2.3,Analytical Geometry,inherited,medium,"If LK = 12 units and 3KN =4SM, determine the l..."
118,11.2.4,Trigonometry,keyword,high,"If it is further given that NL = 16 units, LS ..."


  11.1: [Analytical Geometry|keyword] In the diagram, chords DE, EF and DF are drawn in the circle with centre O.  KFC is a tangent to the circle at F.  J. \ 
  11.2: [Analytical Geometry|keyword] In the diagram, PK is a tangent to the circle at K. Chord LS is produced to P. N and M are points on KP and SP _ respect
  11.2.1: [Euclidean Geometry|keyword] Prove, giving reasons, that: @) &,=NNIL (4) (b) KLMN is acyclic quadrilateral
  11.2.2: [Analytical Geometry|inherited] Prove, giving reasons, that ALKN ||| AKSM.


In [14]:
from pathlib import Path
from datetime import datetime, timezone
import re
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
SEG_DIR = PROJECT_ROOT / "data" / "interim" / "segmented" / "2021"
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2021"
MAP_DIR.mkdir(parents=True, exist_ok=True)

exams = pd.read_csv(SEG_DIR / "exam_questions_2021.csv")


def norm(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).lower()
    return s.replace("θ", "theta").replace("π", "pi")


def map_topic_v2021(text: str, paper: str = ""):
    t = norm(text)
    paper = str(paper).upper()

    # Calculus — strong cues
    if re.search(
        r"f'\s*\(|dy/dx|d/dx|derivative|differentiate|first principles|"
        r"second derivative|concave|inflection|stationary|integrat|"
        r"gradient of the curve|turning point.*f\(|cubic",
        t,
    ):
        return "Calculus", "symbolic", "D_CALC", "high"

    # Calculus — optimisation structure (P1 Q11 style)
    if re.search(
        r"(cost|profit|area|volume|speed|distance).{0,60}(minimum|maximum|minimise|minimize|maximise|maximize|as low as possible|as high as possible)"
        r"|(minimum|maximum|minimise|minimize).{0,60}(cost|profit|area|volume|speed)",
        t,
    ):
        return "Calculus", "structural_opt", "D_CALC_OPT", "high"

    # Euclidean — before AG inheritance targets
    if re.search(
        r"cyclic\s*quad|cyclic quadrilateral|theorem|"
        r"tangent[- ]chord|angle in (the )?semicircle|"
        r"prove that|calculate.*with reasons|give reasons|"
        r"euclidean|similar triangles|proportional",
        t,
    ):
        return "Euclidean Geometry", "keyword", "D_EUCL", "high"

    if re.search(
        r"\bsin\b|\bcos\b|\btan\b|\bsec\b|\bcosec\b|\bcot\b|"
        r"identit|reduction|general solution|trigonomet|theta",
        t,
    ):
        return "Trigonometry", "keyword", "D_TRIG", "high"

    if re.search(
        r"gradient of|equation of the line|midpoint|distance formula|"
        r"circle with centre|radius of|analytical|coordinates of the centre|"
        r"\bcircles?\b|\(x\s*[−\-–]",
        t,
    ):
        return "Analytical Geometry", "keyword", "D_AGEO", "high"

    if re.search(
        r"boxplot|box-and-whisker|standard deviation|variance|ogive|"
        r"histogram|quartile|interquartile|percentile|scatter",
        t,
    ):
        return "Statistics", "keyword", "D_STAT", "high"

    if re.search(
        r"probability|independent|mutually exclusive|tree diagram|venn|p\s*\(|random",
        t,
    ):
        return "Probability", "keyword", "D_PROB", "high"

    if re.search(
        r"arithmetic|geometric|sequence|series|common difference|"
        r"common ratio|constant ratio|consecutive terms|sigma|"
        r"\bterm of\b|first term",
        t,
    ):
        return "Number Patterns & Sequences", "keyword", "D_SEQ", "high"

    if re.search(
        r"compound interest|simple interest|present value|future value|"
        r"annuity|depreciation|effective rate|nominal|loan|invest",
        t,
    ):
        return "Finance", "keyword", "D_FIN", "high"

    if re.search(
        r"parabola|hyperbola|exponential|logarithmic|asymptote|"
        r"inverse function|axis of symmetry|turning point|sketch|"
        r"x-intercept|y-intercept|range of|domain",
        t,
    ):
        return "Functions & Graphs", "keyword", "D_FUNC", "high"

    if re.search(
        r"quadratic|factoris|factoriz|simultaneous|inequalit|surd|"
        r"exponent|logarithm|nature of the roots|discriminant|"
        r"solve for|=\s*0|show that|determine the values of x",
        t,
    ):
        return "Algebra & Equations", "keyword", "D_ALG", "high"

    if re.search(r"[a-z0-9\)\]]\s*=\s*[a-z0-9\-\+]", t) and len(t) < 80 and paper == "P1":
        return "Algebra & Equations", "ocr_equation", "D_ALG_EQ", "medium"

    if len(t) < 20:
        return "Unmapped", "prior_weak", "D_SHORT", "low"

    return "Unmapped", "no_match", "D_NONE", "low"


def inherit_topic_safe(group: pd.DataFrame) -> pd.DataFrame:
    known = group[group["topic"] != "Unmapped"]
    if known.empty:
        return group
    for forced in ["Euclidean Geometry", "Calculus"]:
        hit = known[
            (known["topic"] == forced)
            & (known["mapping_confidence"].isin(["high", "medium"]))
        ]
        if len(hit):
            winner = forced
            break
    else:
        high = known[known["mapping_confidence"] == "high"]
        base = high if len(high) else known
        winner = base["topic"].value_counts().index[0]
    mask = group["topic"] == "Unmapped"
    group.loc[mask, "topic"] = winner
    group.loc[mask, "mapping_method"] = "inherited"
    group.loc[mask, "rule_id"] = "D_INHERIT_Q_SAFE"
    group.loc[mask, "mapping_confidence"] = "medium"
    return group


rows = []
for _, r in exams.iterrows():
    topic, method, rule_id, conf = map_topic_v2021(
        r.get("question_text", ""), r.get("paper", "")
    )
    row = r.to_dict()
    row.update({
        "topic": topic,
        "mapping_method": method,
        "rule_id": rule_id,
        "mapping_confidence": conf,
        "taxonomy_version": "v1_frozen",
        "extraction_method": "ocr_text",
        "vision_description": None,
        "vision_verification_status": None,
        "diagram_present": False,
        "load_bearing_diagram": False,
        "data_quality": "text_usable",
        "mapped_at": datetime.now(timezone.utc).isoformat(),
    })
    rows.append(row)

mapped = pd.DataFrame(rows)
mapped = mapped.groupby(["paper", "question_number"], group_keys=False).apply(inherit_topic_safe)

# Question-level force for known thin-OCR blocks (documented)
FORCE = {
    ("P1", 9): "Calculus",
    ("P1", 10): "Calculus",
    ("P1", 11): "Calculus",
    ("P2", 9): "Euclidean Geometry",
    ("P2", 10): "Euclidean Geometry",
    ("P2", 11): "Euclidean Geometry",
}

def force_q(row):
    try:
        key = (str(row["paper"]), int(float(row["question_number"])))
    except Exception:
        return row
    if key in FORCE and row["topic"] != FORCE[key]:
        row["topic"] = FORCE[key]
        row["mapping_method"] = "question_force_2021"
        row["rule_id"] = "D_FORCE_Q"
        row["mapping_confidence"] = "high"
    return row

mapped = mapped.apply(force_q, axis=1)

print("Topic counts:")
print(mapped["topic"].value_counts())
print()
print("Calculus:", int((mapped["topic"] == "Calculus").sum()))
print("Euclidean:", int((mapped["topic"] == "Euclidean Geometry").sum()))
print("Coverage:", f"{(mapped['topic'] != 'Unmapped').mean():.1%}")

out = MAP_DIR / "question_topic_map_2021.csv"
mapped.to_csv(out, index=False)
print("Saved:", out)

Topic counts:
topic
Analytical Geometry            26
Algebra & Equations            15
Trigonometry                   14
Statistics                     13
Functions & Graphs             12
Euclidean Geometry              9
Number Patterns & Sequences     7
Calculus                        7
Probability                     6
Finance                         5
Unmapped                        5
Name: count, dtype: int64

Calculus: 7
Euclidean: 9
Coverage: 95.8%
Saved: C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\mapped\2021\question_topic_map_2021.csv


In [17]:
from pathlib import Path
from datetime import datetime, timezone
import re
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
SEG_DIR = PROJECT_ROOT / "data" / "interim" / "segmented" / "2021"
MAP_DIR = PROJECT_ROOT / "data" / "processed" / "mapped" / "2021"
MAP_DIR.mkdir(parents=True, exist_ok=True)

exams = pd.read_csv(SEG_DIR / "exam_questions_2021.csv")
print("exams columns:", list(exams.columns))
assert "paper" in exams.columns, "Segment file missing paper"


def norm(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    return str(s).lower().replace("θ", "theta").replace("π", "pi")


def map_topic_v2021(text: str, paper: str = ""):
    t = norm(text)
    paper = str(paper).upper()

    if re.search(
        r"f'\s*\(|dy/dx|d/dx|derivative|differentiate|first principles|"
        r"second derivative|concave|inflection|stationary|integrat|"
        r"gradient of the curve|turning point.*f\(|cubic",
        t,
    ):
        return "Calculus", "symbolic", "D_CALC", "high"

    if re.search(
        r"(cost|profit|area|volume|speed|distance).{0,60}(minimum|maximum|minimise|minimize|maximise|maximize|as low as possible|as high as possible)"
        r"|(minimum|maximum|minimise|minimize).{0,60}(cost|profit|area|volume|speed)",
        t,
    ):
        return "Calculus", "structural_opt", "D_CALC_OPT", "high"

    if re.search(
        r"cyclic\s*quad|cyclic quadrilateral|theorem|"
        r"tangent[- ]chord|angle in (the )?semicircle|"
        r"prove that|calculate.*with reasons|give reasons|"
        r"euclidean|similar triangles|proportional",
        t,
    ):
        return "Euclidean Geometry", "keyword", "D_EUCL", "high"

    if re.search(
        r"\bsin\b|\bcos\b|\btan\b|\bsec\b|\bcosec\b|\bcot\b|"
        r"identit|reduction|general solution|trigonomet|theta",
        t,
    ):
        return "Trigonometry", "keyword", "D_TRIG", "high"

    if re.search(
        r"gradient of|equation of the line|midpoint|distance formula|"
        r"circle with centre|radius of|analytical|coordinates of the centre|"
        r"\bcircles?\b|\(x\s*[−\-–]",
        t,
    ):
        return "Analytical Geometry", "keyword", "D_AGEO", "high"

    if re.search(
        r"boxplot|box-and-whisker|standard deviation|variance|ogive|"
        r"histogram|quartile|interquartile|percentile|scatter",
        t,
    ):
        return "Statistics", "keyword", "D_STAT", "high"

    if re.search(
        r"probability|independent|mutually exclusive|tree diagram|venn|p\s*\(|random",
        t,
    ):
        return "Probability", "keyword", "D_PROB", "high"

    if re.search(
        r"arithmetic|geometric|sequence|series|common difference|"
        r"common ratio|constant ratio|consecutive terms|sigma|"
        r"\bterm of\b|first term",
        t,
    ):
        return "Number Patterns & Sequences", "keyword", "D_SEQ", "high"

    if re.search(
        r"compound interest|simple interest|present value|future value|"
        r"annuity|depreciation|effective rate|nominal|loan|invest",
        t,
    ):
        return "Finance", "keyword", "D_FIN", "high"

    if re.search(
        r"parabola|hyperbola|exponential|logarithmic|asymptote|"
        r"inverse function|axis of symmetry|turning point|sketch|"
        r"x-intercept|y-intercept|range of|domain",
        t,
    ):
        return "Functions & Graphs", "keyword", "D_FUNC", "high"

    if re.search(
        r"quadratic|factoris|factoriz|simultaneous|inequalit|surd|"
        r"exponent|logarithm|nature of the roots|discriminant|"
        r"solve for|=\s*0|show that|determine the values of x",
        t,
    ):
        return "Algebra & Equations", "keyword", "D_ALG", "high"

    if re.search(r"[a-z0-9\)\]]\s*=\s*[a-z0-9\-\+]", t) and len(t) < 80 and paper == "P1":
        return "Algebra & Equations", "ocr_equation", "D_ALG_EQ", "medium"

    if len(t) < 20:
        return "Unmapped", "prior_weak", "D_SHORT", "low"
    return "Unmapped", "no_match", "D_NONE", "low"


rows = []
for _, r in exams.iterrows():
    topic, method, rule_id, conf = map_topic_v2021(r.get("question_text", ""), r.get("paper", ""))
    rows.append({
        "document_id": r.get("document_id"),
        "year": 2021,
        "paper": r["paper"],
        "question_number": r["question_number"],
        "subquestion": r["subquestion"],
        "marks": r.get("marks"),
        "question_text": r.get("question_text"),
        "source_type": r.get("source_type", "exam"),
        "segmentation_status": r.get("segmentation_status"),
        "topic": topic,
        "mapping_method": method,
        "rule_id": rule_id,
        "mapping_confidence": conf,
        "taxonomy_version": "v1_frozen",
        "extraction_method": "ocr_text",
        "vision_description": None,
        "vision_verification_status": None,
        "diagram_present": False,
        "load_bearing_diagram": False,
        "data_quality": "text_usable",
        "mapped_at": datetime.now(timezone.utc).isoformat(),
    })

mapped = pd.DataFrame(rows)

# Inheritance without dropping columns
parts = []
for (paper, qnum), g in mapped.groupby(["paper", "question_number"], sort=False):
    g = g.copy()
    known = g[g["topic"] != "Unmapped"]
    if known.empty:
        parts.append(g)
        continue
    winner = None
    for forced in ["Euclidean Geometry", "Calculus"]:
        hit = known[(known["topic"] == forced) & (known["mapping_confidence"].isin(["high", "medium"]))]
        if len(hit):
            winner = forced
            break
    if winner is None:
        high = known[known["mapping_confidence"] == "high"]
        base = high if len(high) else known
        winner = base["topic"].value_counts().index[0]
    mask = g["topic"] == "Unmapped"
    g.loc[mask, "topic"] = winner
    g.loc[mask, "mapping_method"] = "inherited"
    g.loc[mask, "rule_id"] = "D_INHERIT_Q_SAFE"
    g.loc[mask, "mapping_confidence"] = "medium"
    parts.append(g)

mapped = pd.concat(parts, ignore_index=True)

FORCE = {
    ("P1", 9): "Calculus",
    ("P1", 10): "Calculus",
    ("P1", 11): "Calculus",
    ("P2", 9): "Euclidean Geometry",
    ("P2", 10): "Euclidean Geometry",
    ("P2", 11): "Euclidean Geometry",
}
for i, row in mapped.iterrows():
    try:
        key = (str(row["paper"]), int(float(row["question_number"])))
    except Exception:
        continue
    if key in FORCE and row["topic"] != FORCE[key]:
        mapped.at[i, "topic"] = FORCE[key]
        mapped.at[i, "mapping_method"] = "question_force_2021"
        mapped.at[i, "rule_id"] = "D_FORCE_Q"
        mapped.at[i, "mapping_confidence"] = "high"

assert "paper" in mapped.columns
print("Topic counts:")
print(mapped["topic"].value_counts())
print("Calculus:", int((mapped["topic"] == "Calculus").sum()))
print("Euclidean:", int((mapped["topic"] == "Euclidean Geometry").sum()))
print("Coverage:", f"{(mapped['topic'] != 'Unmapped').mean():.1%}")

out = MAP_DIR / "question_topic_map_2021.csv"
mapped.to_csv(out, index=False)
print("Saved columns:", list(mapped.columns))
print("Saved:", out)

for paper, q in [("P1", 9), ("P1", 11), ("P2", 9), ("P2", 11)]:
    m = mapped[
        (mapped["paper"].astype(str) == paper)
        & (mapped["question_number"].astype(str).str.replace(r"\.0$", "", regex=True) == str(q))
    ]
    print(paper, f"Q{q}", "rows=", len(m), "→", m["topic"].value_counts().to_dict())

exams columns: ['document_id', 'year', 'paper', 'question_number', 'subquestion', 'marks', 'question_text', 'source_type', 'segmentation_status']
Topic counts:
topic
Analytical Geometry            20
Euclidean Geometry             19
Algebra & Equations            13
Trigonometry                   13
Statistics                     13
Functions & Graphs             12
Calculus                       11
Number Patterns & Sequences     7
Probability                     6
Finance                         5
Name: count, dtype: int64
Calculus: 11
Euclidean: 19
Coverage: 100.0%
Saved columns: ['document_id', 'year', 'paper', 'question_number', 'subquestion', 'marks', 'question_text', 'source_type', 'segmentation_status', 'topic', 'mapping_method', 'rule_id', 'mapping_confidence', 'taxonomy_version', 'extraction_method', 'vision_description', 'vision_verification_status', 'diagram_present', 'load_bearing_diagram', 'data_quality', 'mapped_at']
Saved: C:\Users\Administrator\Desktop\Matric-Maths-Ex

In [18]:
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
docs = ROOT / "docs"
docs.mkdir(exist_ok=True)
path = docs / "roadmap.md"

block = """
## Technical debt register (2021 pilot)

- [ ] `question_force_2021` in 2021 mapping — remove when taxonomy v2 lands
- [ ] Re-map 2021 with taxonomy v2 (segmentation preserved)
- [ ] Decide Path A vs Path B for 2020–2014 before opening 2020

### Path choice (lock before 2020)

**Path A — Pure deferral (recommended)**  
2020→2014: extraction + segmentation only. No mapping, no force, no sample accept.  
Map all years once with CAPS-aligned taxonomy v2.

**Path B — Provisional mapping with patches**  
Each year may need `question_force_YYYY`. Higher cost; more debt to remove at rebuild.

**Decision:** _pending_
"""

prev = path.read_text(encoding="utf-8") if path.exists() else "# MatricMath Roadmap\n"
if "question_force_2021" not in prev:
    path.write_text(prev.rstrip() + "\n" + block, encoding="utf-8")
print("Updated", path)

Updated C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\roadmap.md


In [19]:
from pathlib import Path
from datetime import datetime, timezone
import re
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
YEAR = 2020

RAW_EXAMS = PROJECT_ROOT / "data" / "raw" / "exams"
RAW_MEMOS = PROJECT_ROOT / "data" / "raw" / "memos"
TEXT_DIR = PROJECT_ROOT / "data" / "interim" / "extracted_text" / str(YEAR)
SEG_DIR = PROJECT_ROOT / "data" / "interim" / "segmented" / str(YEAR)
PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / str(YEAR)

for d in [TEXT_DIR, SEG_DIR, PILOT_DOC]:
    d.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "p1_exam": RAW_EXAMS / f"{YEAR}_nov_p1_exam_maths.pdf",
    "p2_exam": RAW_EXAMS / f"{YEAR}_nov_p2_exam_maths.pdf",
    "p1_memo": RAW_MEMOS / f"{YEAR}_nov_p1_memo_maths.pdf",
    "p2_memo": RAW_MEMOS / f"{YEAR}_nov_p2_memo_maths.pdf",
}

print("YEAR:", YEAR)
for k, p in SOURCES.items():
    ok = p.exists() and p.stat().st_size > 0
    size = p.stat().st_size if p.exists() else 0
    print(f"  {k}: {'OK' if ok else 'MISSING'}  {size:>10}  {p.name}")

# If MISSING, list matches:
print("\nMatching files:")
for p in sorted(list(RAW_EXAMS.glob(f"*{YEAR}*")) + list(RAW_MEMOS.glob(f"*{YEAR}*"))):
    print(" ", p)

YEAR: 2020
  p1_exam: OK      333635  2020_nov_p1_exam_maths.pdf
  p2_exam: OK      545835  2020_nov_p2_exam_maths.pdf
  p1_memo: OK      760847  2020_nov_p1_memo_maths.pdf
  p2_memo: OK      906808  2020_nov_p2_memo_maths.pdf

Matching files:
  C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\exams\2020_nov_p1_exam_maths.pdf
  C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\exams\2020_nov_p2_exam_maths.pdf
  C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\memos\2020_nov_p1_memo_maths.pdf
  C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\memos\2020_nov_p2_memo_maths.pdf


In [20]:
import pdfplumber

rows = []
for name, path in SOURCES.items():
    if not path.exists():
        rows.append({"year": YEAR, "doc": name, "exists": False})
        continue
    with pdfplumber.open(path) as pdf:
        n = len(pdf.pages)
        sample = ""
        for page in pdf.pages[: min(3, n)]:
            sample += page.extract_text() or ""
        cpp = len(sample) / max(1, min(3, n))
        rows.append({
            "year": YEAR,
            "doc": name,
            "exists": True,
            "pages": n,
            "chars_per_page": round(cpp, 1),
            "has_text_layer": cpp > 200,
            "ocr_required": cpp <= 200,
        })

cls = pd.DataFrame(rows)
cls.to_csv(TEXT_DIR / f"ocr_classification_{YEAR}.csv", index=False)
display(cls)
if cls.get("ocr_required", pd.Series(dtype=bool)).fillna(False).any():
    print("OCR required for at least one doc — use Cell 3b for those.")

,year,doc,exists,pages,chars_per_page,has_text_layer,ocr_required
0,2020,p1_exam,True,11,0.0,False,True
1,2020,p2_exam,True,15,0.0,False,True
2,2020,p1_memo,True,18,862.7,True,False
3,2020,p2_memo,True,27,965.3,True,False


OCR required for at least one doc — use Cell 3b for those.


In [21]:
def extract_pdfplumber(path: Path) -> str:
    parts = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, 1):
            parts.append(f"\n===== Page {i} =====\n{page.extract_text() or ''}")
    return "\n".join(parts)

for name in ["p1_memo", "p2_memo"]:
    if not SOURCES[name].exists():
        print("SKIP", name)
        continue
    if cls.loc[cls["doc"] == name, "ocr_required"].iloc[0]:
        print("SKIP memo needs OCR:", name)
        continue
    text = extract_pdfplumber(SOURCES[name])
    out = TEXT_DIR / f"{name}.txt"
    out.write_text(text, encoding="utf-8")
    print(name, "chars=", len(text))

p1_memo chars= 13592
p2_memo chars= 18958


In [22]:
import os
from pdf2image import convert_from_path
import pytesseract

POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"
TESSDATA = r"C:\Users\Administrator\tessdata"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
os.environ["TESSDATA_PREFIX"] = TESSDATA
OCR_CONFIG = r"--tessdata-dir C:\Users\Administrator\tessdata --psm 6"

for name in ["p1_exam", "p2_exam"]:
    need = True
    if name in cls["doc"].values:
        need = bool(cls.loc[cls["doc"] == name, "ocr_required"].iloc[0])
    if not need:
        # still extract with pdfplumber if text layer
        text = extract_pdfplumber(SOURCES[name])
        (TEXT_DIR / f"{name}.txt").write_text(text, encoding="utf-8")
        print(name, "pdfplumber chars=", len(text))
        continue
    print(f"OCR {name}...")
    pages = convert_from_path(str(SOURCES[name]), dpi=300, poppler_path=POPPLER_PATH)
    parts = []
    for i, img in enumerate(pages, 1):
        t = pytesseract.image_to_string(img, lang="eng", config=OCR_CONFIG)
        parts.append(f"\n===== Page {i} =====\n{t}")
        print(f"  page {i}/{len(pages)} chars={len(t)}")
    full = "\n".join(parts)
    (TEXT_DIR / f"{name}.txt").write_text(full, encoding="utf-8")
    print(f"  saved chars={len(full)}")

OCR p1_exam...
  page 1/11 chars=329
  page 2/11 chars=890
  page 3/11 chars=972
  page 4/11 chars=531
  page 5/11 chars=692
  page 6/11 chars=499
  page 7/11 chars=1336
  page 8/11 chars=620
  page 9/11 chars=1419
  page 10/11 chars=697
  page 11/11 chars=763
  saved chars=8980
OCR p2_exam...
  page 1/15 chars=338
  page 2/15 chars=836
  page 3/15 chars=1220
  page 4/15 chars=2053
  page 5/15 chars=718
  page 6/15 chars=796
  page 7/15 chars=576
  page 8/15 chars=787
  page 9/15 chars=728
  page 10/15 chars=349
  page 11/15 chars=432
  page 12/15 chars=347
  page 13/15 chars=372
  page 14/15 chars=631
  page 15/15 chars=779
  saved chars=11282


In [23]:
def clean_text(t: str) -> str:
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    return re.sub(r"\n{3,}", "\n\n", t)


def segment_exam(text: str, document_id: str, paper: str, year: int) -> pd.DataFrame:
    text = clean_text(text)
    q_pat = re.compile(r"(?i)(?:^|\n)\s*QUESTION\s*(\d{1,2})\b")
    matches = list(q_pat.finditer(text))
    records = []
    for i, m in enumerate(matches):
        qnum = int(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        sub_pat = re.compile(
            r"(?m)(?P<sub>\d+\.\d+(?:\.\d+)*)\s+(?P<body>.*?)(?=(?:\n\s*\d+\.\d+(?:\.\d+)*\s+)|\Z)",
            re.DOTALL,
        )
        subs = list(sub_pat.finditer(block))
        if not subs:
            records.append({
                "year": year,
                "document_id": document_id,
                "paper": paper,
                "question_number": qnum,
                "subquestion": str(qnum),
                "marks": None,
                "question_text": block[:2000],
                "segmentation_status": "no_sub_markers",
            })
            continue
        for s in subs:
            body = s.group("body").strip()
            marks = None
            mm = re.search(r"[\(\[]\s*(\d{1,2})\s*[\)\]]\s*$", body)
            if mm:
                marks = int(mm.group(1))
                body = body[: mm.start()].strip()
            if marks is not None and marks > 15:
                marks = None
            records.append({
                "year": year,
                "document_id": document_id,
                "paper": paper,
                "question_number": qnum,
                "subquestion": s.group("sub").strip(),
                "marks": marks,
                "question_text": body[:2000],
                "segmentation_status": "ok" if marks is not None else "missing_marks",
            })
    return pd.DataFrame(records)


all_rows = []
for paper, fname, did in [
    ("P1", "p1_exam.txt", f"{YEAR}_nov_p1_exam_maths"),
    ("P2", "p2_exam.txt", f"{YEAR}_nov_p2_exam_maths"),
]:
    text = (TEXT_DIR / fname).read_text(encoding="utf-8", errors="replace")
    df = segment_exam(text, did, paper, YEAR)
    print(f"{paper}: rows={len(df)} main_q={df['question_number'].nunique()} marks_sum={df['marks'].sum(min_count=1)}")
    all_rows.append(df)

exam_df = pd.concat(all_rows, ignore_index=True)

# Path A schema only
SEGMENTATION_COLUMNS = [
    "year", "document_id", "paper", "question_number",
    "subquestion", "marks", "question_text", "segmentation_status",
]
exam_df = exam_df[SEGMENTATION_COLUMNS]

forbidden = {
    "topic", "subtopic", "skill", "mapping_method", "mapping_confidence",
    "extraction_method", "diagram_present", "vision_description",
}
leaked = forbidden & set(exam_df.columns)
if leaked:
    raise SystemExit(f"Path A violation: {leaked}")

out = SEG_DIR / f"exam_questions_{YEAR}.csv"
exam_df.to_csv(out, index=False)
print("Wrote", out, "rows=", len(exam_df))
print("Path A check: PASS")
display(exam_df.head(12))

P1: rows=55 main_q=11 marks_sum=108.0
P2: rows=52 main_q=10 marks_sum=78.0
Wrote C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\interim\segmented\2020\exam_questions_2020.csv rows= 107
Path A check: PASS


,year,document_id,paper,question_number,subquestion,marks,question_text,segmentation_status
0,2020,2020_nov_p1_exam_maths,P1,1,1.1,NaN,Solve for x:,missing_marks
1,2020,2020_nov_p1_exam_maths,P1,1,1.1.1,2.0,x’ —-6x =0,ok
2,2020,2020_nov_p1_exam_maths,P1,1,1.1.2,3.0,x’ +10x+8=0 (correct to TWO decimal places),ok
3,2020,2020_nov_p1_exam_maths,P1,1,1.1.3,3.0,(l—x\{x+2)<0,ok
4,2020,2020_nov_p1_exam_maths,P1,1,1.1.4,5.0,Vx+18 =x-2,ok
5,2020,2020_nov_p1_exam_maths,P1,1,1.2,6.0,Solve simultaneously for x and y:\nx+y=3 and 2...,ok
6,2020,2020_nov_p1_exam_maths,P1,1,1.3,NaN,"If is the largest integer for which n° < 5°”, ...",missing_marks
7,2020,2020_nov_p1_exam_maths,P1,2,2.1,4.0,73x%3;y3-I1 ; ... isan arithmetic sequence. De...,ok
8,2020,2020_nov_p1_exam_maths,P1,2,2.2,NaN,Given the quadratic number pattern: —3 ; 6 ; 2...,missing_marks
9,2020,2020_nov_p1_exam_maths,P1,2,2.2.1,4.0,Determine the general term of the pattern in t...,ok


In [24]:
marker = PILOT_DOC / "path_a_freeze.md"
marker.write_text(f"""# {YEAR} — Path A freeze

Stage: extraction + segmentation only.
Mapping: NOT DONE. Deferred to taxonomy v2.

Artefacts:
- data/interim/extracted_text/{YEAR}/
- data/interim/segmented/{YEAR}/exam_questions_{YEAR}.csv

No topic, no sample verification, no vision, no question_force.

Do not run N05/N06/N07 on this year until taxonomy v2 mapping.
Frozen: {datetime.now(timezone.utc).isoformat()}
""", encoding="utf-8")
print("Wrote", marker)

Wrote C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2020\path_a_freeze.md


In [25]:
import os
import re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import pdfplumber

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
RAW_EXAMS = PROJECT_ROOT / "data" / "raw" / "exams"
RAW_MEMOS = PROJECT_ROOT / "data" / "raw" / "memos"

# Years to process in order. Comment/uncomment as needed.
YEARS = [2019, 2018, 2017, 2016, 2015, 2014]
# To run one year only (for debugging): YEARS = [2019]

# OCR tool paths — only used when classification flags a doc
POPPLER_PATH = r"C:\Users\Administrator\Downloads\Release-24.08.0-0\poppler-24.08.0\Library\bin"
TESSDATA = r"C:\Users\Administrator\tessdata"
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("Years queued:", YEARS)
print("Exams dir   :", RAW_EXAMS, "exists:", RAW_EXAMS.exists())
print("Memos dir   :", RAW_MEMOS, "exists:", RAW_MEMOS.exists())

Years queued: [2019, 2018, 2017, 2016, 2015, 2014]
Exams dir   : C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\exams exists: True
Memos dir   : C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\raw\memos exists: True


In [26]:
# ---- helpers ----

def doc_paths(year: int) -> dict:
    return {
        "p1_exam": RAW_EXAMS / f"{year}_nov_p1_exam_maths.pdf",
        "p2_exam": RAW_EXAMS / f"{year}_nov_p2_exam_maths.pdf",
        "p1_memo": RAW_MEMOS / f"{year}_nov_p1_memo_maths.pdf",
        "p2_memo": RAW_MEMOS / f"{year}_nov_p2_memo_maths.pdf",
    }


def classify_doc(path: Path) -> dict:
    if not path.exists() or path.stat().st_size == 0:
        return {"exists": False, "ocr_required": False, "pages": 0, "chars_per_page": 0.0}
    with pdfplumber.open(path) as pdf:
        n = len(pdf.pages)
        sample = ""
        for page in pdf.pages[: min(3, n)]:
            sample += page.extract_text() or ""
        cpp = len(sample) / max(1, min(3, n))
    return {
        "exists": True,
        "pages": n,
        "chars_per_page": round(cpp, 1),
        "has_text_layer": cpp > 200,
        "ocr_required": cpp <= 200,
    }


def extract_text_layer(path: Path) -> str:
    parts = []
    with pdfplumber.open(path) as pdf:
        for i, page in enumerate(pdf.pages, 1):
            parts.append(f"\n===== Page {i} =====\n{page.extract_text() or ''}")
    return "\n".join(parts)


def extract_with_ocr(path: Path) -> str:
    import pytesseract
    from pdf2image import convert_from_path
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD
    os.environ["TESSDATA_PREFIX"] = TESSDATA
    cfg = r"--tessdata-dir " + TESSDATA + r" --psm 6"
    pages = convert_from_path(str(path), dpi=300, poppler_path=POPPLER_PATH)
    parts = []
    for i, img in enumerate(pages, 1):
        t = pytesseract.image_to_string(img, lang="eng", config=cfg)
        parts.append(f"\n===== Page {i} =====\n{t}")
    return "\n".join(parts)


def clean_text(t: str) -> str:
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    return re.sub(r"\n{3,}", "\n\n", t)


def segment_exam(text: str, document_id: str, paper: str, year: int) -> pd.DataFrame:
    text = clean_text(text)
    q_pat = re.compile(r"(?i)(?:^|\n)\s*QUESTION\s*(\d{1,2})\b")
    matches = list(q_pat.finditer(text))
    records = []
    for i, m in enumerate(matches):
        qnum = int(m.group(1))
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()
        sub_pat = re.compile(
            r"(?m)(?P<sub>\d+\.\d+(?:\.\d+)*)\s+(?P<body>.*?)(?=(?:\n\s*\d+\.\d+(?:\.\d+)*\s+)|\Z)",
            re.DOTALL,
        )
        subs = list(sub_pat.finditer(block))
        if not subs:
            records.append({
                "year": year, "document_id": document_id, "paper": paper,
                "question_number": qnum, "subquestion": str(qnum),
                "marks": None, "question_text": block[:2000],
                "segmentation_status": "no_sub_markers",
            })
            continue
        for s in subs:
            body = s.group("body").strip()
            marks = None
            mm = re.search(r"[\(\[]\s*(\d{1,2})\s*[\)\]]\s*$", body)
            if mm:
                marks = int(mm.group(1))
                body = body[: mm.start()].strip()
            if marks is not None and marks > 15:
                marks = None
            records.append({
                "year": year, "document_id": document_id, "paper": paper,
                "question_number": qnum, "subquestion": s.group("sub").strip(),
                "marks": marks, "question_text": body[:2000],
                "segmentation_status": "ok" if marks is not None else "missing_marks",
            })
    return pd.DataFrame(records)


SEGMENTATION_COLUMNS = [
    "year", "document_id", "paper", "question_number",
    "subquestion", "marks", "question_text", "segmentation_status",
]
FORBIDDEN = {
    "topic", "subtopic", "skill", "mapping_method", "mapping_confidence",
    "extraction_method", "diagram_present", "vision_description",
}


# ---- main loop ----

summary_rows = []

for YEAR in YEARS:
    print(f"\n{'='*70}\nYEAR {YEAR}\n{'='*70}")

    TEXT_DIR = PROJECT_ROOT / "data" / "interim" / "extracted_text" / str(YEAR)
    SEG_DIR  = PROJECT_ROOT / "data" / "interim" / "segmented" / str(YEAR)
    PILOT_DOC = PROJECT_ROOT / "docs" / "pilot" / str(YEAR)
    for d in (TEXT_DIR, SEG_DIR, PILOT_DOC):
        d.mkdir(parents=True, exist_ok=True)

    sources = doc_paths(YEAR)

    # --- classification ---
    cls_rows = []
    for name, path in sources.items():
        info = classify_doc(path)
        cls_rows.append({"year": YEAR, "doc": name, **info})
    cls = pd.DataFrame(cls_rows)
    cls.to_csv(TEXT_DIR / f"ocr_classification_{YEAR}.csv", index=False)
    print(cls.to_string(index=False))

    missing = cls.loc[~cls["exists"], "doc"].tolist()
    if missing:
        print(f"  MISSING sources for {YEAR}: {missing} — skipping year")
        summary_rows.append({
            "year": YEAR, "status": "skipped_missing_sources",
            "p1_rows": 0, "p2_rows": 0,
            "p1_marks": 0, "p2_marks": 0, "ocr_used": False,
        })
        continue

    # --- extraction (per doc: text-layer or OCR) ---
    ocr_used_this_year = False
    for name, path in sources.items():
        need_ocr = bool(cls.loc[cls["doc"] == name, "ocr_required"].iloc[0])
        out = TEXT_DIR / f"{name}.txt"
        if out.exists() and out.stat().st_size > 5000:
            print(f"  SKIP extract {name}: text already present")
            continue
        if need_ocr:
            print(f"  OCR  {name} ...")
            try:
                text = extract_with_ocr(path)
                ocr_used_this_year = True
            except Exception as e:
                print(f"    OCR FAILED for {name}: {e}")
                continue
        else:
            print(f"  TEXT {name} ...")
            text = extract_text_layer(path)
        out.write_text(text, encoding="utf-8")
        print(f"    saved chars={len(text)}")

    # --- segmentation ---
    all_rows = []
    for paper, fname, did in [
        ("P1", "p1_exam.txt", f"{YEAR}_nov_p1_exam_maths"),
        ("P2", "p2_exam.txt", f"{YEAR}_nov_p2_exam_maths"),
    ]:
        tpath = TEXT_DIR / fname
        if not tpath.exists():
            print(f"  no text for {paper}; skipping")
            continue
        text = tpath.read_text(encoding="utf-8", errors="replace")
        df = segment_exam(text, did, paper, YEAR)
        all_rows.append(df)

    if not all_rows:
        print(f"  {YEAR}: no segments produced")
        summary_rows.append({
            "year": YEAR, "status": "no_segments",
            "p1_rows": 0, "p2_rows": 0,
            "p1_marks": 0, "p2_marks": 0, "ocr_used": ocr_used_this_year,
        })
        continue

    exam_df = pd.concat(all_rows, ignore_index=True)
    exam_df = exam_df[SEGMENTATION_COLUMNS]

    leaked = FORBIDDEN & set(exam_df.columns)
    if leaked:
        raise SystemExit(f"Path A violation in {YEAR}: {leaked}")

    out = SEG_DIR / f"exam_questions_{YEAR}.csv"
    exam_df.to_csv(out, index=False)

    p1 = exam_df[exam_df["paper"] == "P1"]
    p2 = exam_df[exam_df["paper"] == "P2"]
    p1_marks = int(p1["marks"].sum(min_count=1) or 0)
    p2_marks = int(p2["marks"].sum(min_count=1) or 0)

    print(f"  P1: rows={len(p1)} main_q={p1['question_number'].nunique()} marks={p1_marks}")
    print(f"  P2: rows={len(p2)} main_q={p2['question_number'].nunique()} marks={p2_marks}")
    print(f"  Wrote {out}")

    # --- freeze marker ---
    marker = PILOT_DOC / "path_a_freeze.md"
    marker.write_text(f"""# {YEAR} — Path A freeze

Stage: extraction + segmentation only.
Mapping: NOT DONE. Deferred to taxonomy v2.

Artefacts:
- data/interim/extracted_text/{YEAR}/
- data/interim/segmented/{YEAR}/exam_questions_{YEAR}.csv

Counts:
- P1: {len(p1)} rows, {p1_marks} marks (from OCR/text; may be partial)
- P2: {len(p2)} rows, {p2_marks} marks (from OCR/text; may be partial)
- OCR used: {ocr_used_this_year}

No topic, no sample verification, no vision, no question_force.
Do not run N05/N06/N07 on this year until taxonomy v2 mapping.
Frozen: {datetime.now(timezone.utc).isoformat()}
""", encoding="utf-8")

    summary_rows.append({
        "year": YEAR, "status": "ok",
        "p1_rows": len(p1), "p2_rows": len(p2),
        "p1_marks": p1_marks, "p2_marks": p2_marks,
        "ocr_used": ocr_used_this_year,
    })


# --- progress log ---
log = pd.DataFrame(summary_rows)
log_path = PROJECT_ROOT / "docs" / "pilot" / "path_a_progress.md"
lines = ["# Path A Progress — 2014 to 2020", "",
         "| Year | Status | OCR | P1 rows | P1 marks | P2 rows | P2 marks |",
         "|------|--------|-----|---------|----------|---------|----------|"]
for _, r in log.iterrows():
    lines.append(
        f"| {r['year']} | {r['status']} | {r['ocr_used']} | "
        f"{r['p1_rows']} | {r['p1_marks']} | {r['p2_rows']} | {r['p2_marks']} |"
    )
log_path.write_text("\n".join(lines), encoding="utf-8")
print("\n" + "="*70)
print(log.to_string(index=False))
print(f"\nProgress log: {log_path}")


YEAR 2019
 year     doc  exists  pages  chars_per_page  has_text_layer  ocr_required
 2019 p1_exam    True     10             0.0           False          True
 2019 p2_exam    True     15             0.0           False          True
 2019 p1_memo    True     18           721.7            True         False
 2019 p2_memo    True     26           911.3            True         False
  OCR  p1_exam ...
    saved chars=9161
  OCR  p2_exam ...
    saved chars=10327
  TEXT p1_memo ...
    saved chars=13748
  TEXT p2_memo ...
    saved chars=20309
  P1: rows=56 main_q=11 marks=122
  P2: rows=45 main_q=10 marks=76
  Wrote C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\interim\segmented\2019\exam_questions_2019.csv

YEAR 2018
 year     doc  exists  pages  chars_per_page  has_text_layer  ocr_required
 2018 p1_exam    True     10             0.0           False          True
 2018 p2_exam    True     16             0.0           False          True
 2018 p1_memo    True     

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
df = pd.read_csv(ROOT / "data/interim/segmented/2014/exam_questions_2014.csv")
p1 = df[df["paper"] == "P1"]

print("Subquestions per main question:")
print(p1.groupby("question_number")["subquestion"].apply(list).to_string())

print("\nRows with segmentation_status != ok:")
print(p1[p1["segmentation_status"] != "ok"][["question_number", "subquestion", "segmentation_status"]])

Subquestions per main question:
question_number
1     [1.1, 1.1.1, 1.1.2, 1.1.3, 1.3]
2                     [2.1, 2.3, 2.4]
3        [3.1, 3.1.2, 3.13, 3.2, 3.3]
4                [4.1, 4.2, 4.3, 4.4]
5                          [5.4, 2.9]
6                     [6.1, 6.2, 6.3]
7                        [7.1, 7.2.1]
8                [8.1, 8.2, 8.3, 8.4]
9                          [9.1, 9.3]
10                 [10.1, 10.2, 10.3]
11                     [11.1, 11.1.2]
12                   [12.1.2, 12.2.1]

Rows with segmentation_status != ok:
    question_number subquestion segmentation_status
0                 1         1.1       missing_marks
4                 1         1.3       missing_marks
7                 2         2.4       missing_marks
12                3         3.3       missing_marks
16                4         4.4       missing_marks
18                5         2.9       missing_marks
21                6         6.3       missing_marks
22                7         7.1       miss

In [4]:
import re
import pandas as pd

def clean_text(t: str) -> str:
    t = t.replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    return re.sub(r"\n{3,}", "\n\n", t)

In [ ]:
def segment_exam(text: str, document_id: str, paper: str, year: int) -> pd.DataFrame:
    """
    Segment a paper into subquestions.

    Rules:
      - Question boundaries detected by 'QUESTION <n>' headers.
      - Subquestions detected by patterns like 1.1, 1.1.1, 1.2.3.
      - Subquestion only accepted if its number starts with the parent question
        number (e.g. '5.4' under Q5 kept; '2.9' under Q5 dropped).
      - Marks extracted from a trailing (n) or [n] at end of body.
      - Marks > 15 discarded as OCR noise.
    """
    text = clean_text(text)

    q_pat = re.compile(r"(?i)(?:^|\n)\s*QUESTION\s*(\d{1,2})\b")
    matches = list(q_pat.finditer(text))

    records = []

    for i, m in enumerate(matches):
        qnum = int(m.group(1))
        qnum_str = str(qnum)
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()

        sub_pat = re.compile(
            r"(?m)(?P<sub>\d+\.\d+(?:\.\d+)*)\s+"
            r"(?P<body>.*?)"
            r"(?=(?:\n\s*\d+\.\d+(?:\.\d+)*\s+)|\Z)",
            re.DOTALL,
        )
        subs = list(sub_pat.finditer(block))

        kept_subs = [
            s for s in subs
            if s.group("sub").strip().startswith(qnum_str + ".")
        ]

        if not kept_subs:
            records.append({
                "year": year,
                "document_id": document_id,
                "paper": paper,
                "question_number": qnum,
                "subquestion": qnum_str,
                "marks": None,
                "question_text": block[:2000],
                "segmentation_status": "no_sub_markers",
            })
            continue
for s in kept_subs:
            sub_label = s.group("sub").strip()
            body = s.group("body").strip()

            marks = None
            mm = re.search(r"[\(\[]\s*(\d{1,2})\s*[\)\]]\s*$", body)
            if mm:
                candidate = int(mm.group(1))
                if candidate <= 15:
                    marks = candidate
                    body = body[: mm.start()].strip()

            records.append({
                "year": year,
                "document_id": document_id,
                "paper": paper,
                "question_number": qnum,
                "subquestion": sub_label,
                "marks": marks,
                "question_text": body[:2000],
                "segmentation_status": "ok" if marks is not None else "missing_marks",
            })

    return pd.DataFrame(records)

In [17]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

SEGMENTATION_COLUMNS = [
    "year", "document_id", "paper", "question_number",
    "subquestion", "marks", "question_text", "segmentation_status",
]

for year in [2020, 2019, 2018, 2017, 2016, 2015, 2014]:
    TEXT_DIR = ROOT / "data" / "interim" / "extracted_text" / str(year)
    SEG_DIR  = ROOT / "data" / "interim" / "segmented" / str(year)

    all_rows = []
    for paper, fname, did in [
        ("P1", "p1_exam.txt", f"{year}_nov_p1_exam_maths"),
        ("P2", "p2_exam.txt", f"{year}_nov_p2_exam_maths"),
    ]:
        tpath = TEXT_DIR / fname
        if not tpath.exists():
            print(f"{year} {paper}: no text file, skipping")
            continue
        text = tpath.read_text(encoding="utf-8", errors="replace")
        df = segment_exam(text, did, paper, year)
        all_rows.append(df)

    if not all_rows:
        print(f"{year}: no rows produced"); continue

    exam_df = pd.concat(all_rows, ignore_index=True)
    exam_df = exam_df[SEGMENTATION_COLUMNS]

    out = SEG_DIR / f"exam_questions_{year}.csv"
    exam_df.to_csv(out, index=False)

    p1 = exam_df[exam_df["paper"] == "P1"]
    p2 = exam_df[exam_df["paper"] == "P2"]
    print(f"{year}: P1={len(p1)}  P2={len(p2)}  total={len(exam_df)}")

2020: P1=55  P2=52  total=107
2019: P1=54  P2=45  total=99
2018: P1=57  P2=67  total=124
2017: P1=44  P2=56  total=100
2016: P1=40  P2=42  total=82
2015: P1=40  P2=38  total=78
2014: P1=36  P2=46  total=82


In [18]:
df = pd.read_csv(ROOT / "data" / "interim" / "segmented" / "2014" / "exam_questions_2014.csv")
p1 = df[df["paper"] == "P1"]
print("Subquestions per main question (P1 2014):")
print(p1.groupby("question_number")["subquestion"].apply(list).to_string())

Subquestions per main question (P1 2014):
question_number
1     [1.1, 1.1.1, 1.1.2, 1.1.3, 1.3]
2                     [2.1, 2.3, 2.4]
3        [3.1, 3.1.2, 3.13, 3.2, 3.3]
4                [4.1, 4.2, 4.3, 4.4]
5                               [5.4]
6                     [6.1, 6.2, 6.3]
7                        [7.1, 7.2.1]
8                [8.1, 8.2, 8.3, 8.4]
9                          [9.1, 9.3]
10                 [10.1, 10.2, 10.3]
11                     [11.1, 11.1.2]
12                   [12.1.2, 12.2.1]


In [21]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")
df = pd.read_csv(ROOT / "data" / "interim" / "segmented" / "2018" / "exam_questions_2018.csv")
p2 = df[df["paper"] == "P2"]

print(f"Rows: {len(p2)}")
print(f"Main questions: {p2['question_number'].nunique()}")
print()
print("Subquestions per main question:")
print(p2.groupby("question_number")["subquestion"].apply(list).to_string())
print()
print("Segmentation status counts:")
print(p2["segmentation_status"].value_counts())

Rows: 67
Main questions: 10

Subquestions per main question:
question_number
1     [1.1, 1.1.1, 1.1.2, 1.1.3, 1.1.4, 1.1.5, 1.1.6...
2                    [2.1, 2.2, 2.2.1, 2.2.2, 2.3, 2.4]
3     [3.1, 3.1.1, 3.1.2, 3.2, 3.3, 3.4, 3.5, 3.5.1,...
4        [4.1, 4.2, 4.3, 4.4, 4.5, 4.5.1, 4.5.2, 4.5.3]
5                  [5.1, 5.1.1, 5.1.2, 5.1.3, 5.2, 5.3]
6                             [6.1, 6.2, 6.3, 6.4, 6.5]
7                                            [7.1, 7.2]
8     [8.1, 8.1.1, 8.1.2, 8.1.3, 8.1.4, 8.1.5, 8.2, ...
9                       [9.1, 9.2, 9.2.1, 9.2.2, 9.2.3]
10         [10.1, 10.1.1, 10.1.2, 10.2, 10.2.1, 10.2.2]

Segmentation status counts:
segmentation_status
ok               39
missing_marks    28
Name: count, dtype: int64


In [22]:
from pathlib import Path
ROOT = Path(r"C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence")

note_degraded = """
Segmentation note (v1.1): parent-prefix filter applied.
Remaining segmentation gaps are OCR fidelity limitations on scanned papers:
- merged dots (e.g. 3.1.3 read as 3.13)
- missing short numeric subquestion markers
- parent question kept without all children
These will be recovered during N04 mapping via memo join.
Not a blocker for Path A.
"""

for year in [2016, 2015, 2014]:
    p = ROOT / "docs" / "pilot" / str(year) / "path_a_freeze.md"
    if p.exists():
        p.write_text(p.read_text(encoding="utf-8") + "\n" + note_degraded, encoding="utf-8")
        print(f"Updated {p}")

Updated C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2016\path_a_freeze.md
Updated C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2015\path_a_freeze.md
Updated C:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\pilot\2014\path_a_freeze.md
